# Used Car Price Analysis
### \-Exploring what affects used car prices and building a predictive model.

**Dataset:** UCI Automobile Dataset  
**Tools:** Python,Pandas,Matplotlib,Seaborn,Scipy,Scikit-learn  
**Author:** Marwan Fathi Beheri    
\----------------------  
This notebook explores relationships between car characteristics(make, engine size, horse power, etc. ) and price, then it explores the developing and exploring of several regression models to predict price using these features.

## 1.Data Loading  
We use UCI Machine Learning Repository's Automobile Dataset, which which contains 205 records of used cars with 26 attributes including quantitative variables (engine size, horse power, etc.), categorical features (make, body-style, etc.), and price.

In [ ]:
import pandas as pd

url='https://archive.ics.uci.edu/ml/machine-learning-databases/autos/imports-85.data'

df=pd.read_csv(url,header=None,na_values='?')

headers = [
    "symboling", "normalized-losses", "make", "fuel-type", "aspiration",
    "num-of-doors", "body-style", "drive-wheels", "engine-location",
    "wheel-base", "length", "width", "height", "curb-weight",
    "engine-type", "num-of-cylinders", "engine-size", "fuel-system",
    "bore", "stroke", "compression-ratio", "horsepower",
    "peak-rpm", "city-mpg", "highway-mpg", "price"
]

df.columns=headers

print(df.head())

print(df.dtypes)

print(df.describe(include='all'))

print(df.info())



The dataset does not include column headers, so we assign them manually based on the dataset's documentation. A quick look at `isna().sum()` below shows several columns with missing values, marked as `?` in the raw data.

In [ ]:
num_of_missing_values=df.isna().sum(axis=0)
print(num_of_missing_values)

### Handling missing values

`normalized-losses`, `bore`, `stroke`, `horsepower`, and `peak-rpm` each have missing values. Since these are all continuous numeric variables, we impute the missing values with the **column mean**. This is a reasonable choice since none of these variables has a high amount of missing values that would make mean imputation misleading.

`num-of-doors` is categorical, so we impute its missing values with the **column mode** instead, since the mean does not make sense for a categorical variable.

`price` is the target variable. Generally speaking, deleting the rows that contain a missing target value is better than imputing them.

In [ ]:
mean_names=['normalized-losses','bore','stroke','horsepower','peak-rpm']

for i in mean_names:
 mean=df[i].mean()
 df[i]=df[i].fillna(mean)

mode_nod=df['num-of-doors'].mode()[0]
df['num-of-doors']=df['num-of-doors'].fillna(mode_nod)

df.dropna(subset=['price'],axis=0,inplace=True)




### Converting 'num-of-cylinders' & 'num-of-doors' into a numeric variable  
Unlike the other text based categorical variables like ('make','body-style', etc.) both `num-of-cylinders` and `num-of-doors` are **ordinal** rather than purely categorical, their values have a numeric order and magnitude, since a higher cylinder count means a more powerful engine. converting them into a integer perserves the meaningful numeric relationship between them and price.

In [ ]:
df['num-of-cylinders']=df['num-of-cylinders'].replace({
  "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "eight": 8,
    "twelve": 12
}).astype(int)
print(df['num-of-cylinders'])

df['num-of-doors']=df['num-of-doors'].replace({
  "two": 2,
  "four": 4,
}).astype(int)
print(df['num-of-doors'])

### One hot-encoding the remaining categorical variables  
The remaining categorical columns `fuel-type`, `aspiration`, `engine-location`, `body-style`, `fuel-system`, `engine-type`, and `drive-wheels` are non ordinal, there is no   meaningful numeric order between these variables, so we one-hot encode them instead. For each variable, we drop one resulting dummy   column(`drop_first=True`) to avoid redundancy, since knowing all but one column is already enough to know which category a row belongs to.  

### Note on `make`  
unlike the other categorical variables, `make` has over 20 unique categories. One hot encoding it would create a large number of sparse columns,  
and coefficients for makes that occur rarely which would be unreliable. Therefore we will keep `make` a categorical variable, and exclude it from  
the predictive regression models.




In [ ]:
oh_names=['fuel-type', 'aspiration', 'engine-location', 'body-style', 'drive-wheels','fuel-system','engine-type']
new_col=[]
for i in oh_names:
 dummy_variable=pd.get_dummies(df[i],drop_first=True)
 new_col.extend(dummy_variable.columns.tolist())                             
 df=pd.concat([df,dummy_variable],axis=1)
print(new_col)

### price distribution across categorical variables  
Before performing a statistical test, we visualize the relationship using a box plot for each categorical variable individually, to see if the unique values in each variable overlap significantly, or are they separated?

In [ ]:
x_names=['fuel-type','engine-location','fuel-system','engine-type','aspiration','body-style', 'drive-wheels']
for i in x_names:
  %matplotlib inline
  import seaborn as sns
  import matplotlib.pyplot as plt
  sns.boxplot(x=i,y='price',data=df)
  plt.xlabel(i)
  plt.ylabel('Price')
  plt.show()


**Observations:** 

**fuel-type:** Gas and diesel overlap significantly. This suggests that `fuel-type` can be a weak predictor. 

**engine-location**:Rear has higher price distribution than front. However, the number of rear engine cars is very small so this should be interpreted with caution 

**aspiration:** Turbo and std overlap significantly. This suggests that `aspiration` can be a weak predictor.  

**fuel-system:** `Low group` (2bbl and 1bbl): Their boxes are low and almost completely separate from the higher groups. They only slightly touch the bottom of idi and spdi.  

`Middle group` (4bbl, spdi, and the lower half of idi): These overlap strongly with each other.  

`High group` (mpfi and idi): Both have high, wide boxes that heavily overlap with each other. They sit above most of the lower systems.  

mfi and spfi are just single horizontal lines (very few data points), so they don’t form real boxes and can’t be properly assessed for overlap.  

**engine-type:** `Low group` (ohc and ohcf): Their boxes are low and almost completely separate from the higher groups. they overlap siginificantly.

`Middle group` (l,rotor, and dohc): These overlap strongly with each other.

`High group` (dohc and bottom half of ohcv): Both have high, boxes that overlap with each other. They sit above most of the lower systems. 

**body-style:** hatchback is the lowest and overlap to a certain degree with higher groups, while sedan and wagon overlap strongly, convertible and hardtop are the most expensive and and overlap with each other , and partially overlap with sedan and wagon

**drive-wheels:** (fwd and fwd) overlap almost completely. rwd is higher than both fwd and 4wd 

## Anova for each categorical variable  
We test the categorical variables that went through a box plot, and if it has a statistically significant relationship with price. For each variable, we group price by its categories and peform a one way anova for all the categories simultaneously.

In [ ]:
from scipy import stats
df_anova=df[['fuel-type','engine-location','fuel-system','engine-type','aspiration','body-style', 'drive-wheels','price']]
df_cat=['fuel-type','engine-location','fuel-system','engine-type','aspiration','body-style', 'drive-wheels']
for i in df_cat:
 grouped=df_anova.groupby(i)['price']
 groups=[group for name,group in grouped]
 anova_result= stats.f_oneway(*groups)
 print(f"{i}: F={anova_result.statistic:.3f} ,P={anova_result.pvalue:.3f}")

**Observations:**  
`fuel-type` is not statistically significant so it will be excluded from the predictive model.  
`fuel-system`, `engine-type`, `body-style`, `drive-wheels`, and `aspiration` are statistically significant so they will be included in the predictive model.  
`engine-location` has a problem where only 3 out of 201 cars have a "rear" engine location. although the result of the anova is statistically significant the outcome is unreliable due to haveing such a small group, so it will be excluded from the predictive model. 


In [ ]:
weak_col=['gas','rear']
new_col.remove('gas')
new_col.remove('rear')
print(new_col)

## Heat map for drive-wheels and body-style  
`drive-wheels` and `body-style` can both have an effect on price that differs from the effect of each of them individually. To look into this we will have to group both variables and find the average of each combination and visualize the result as a heat map

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
df_group=df[['drive-wheels','body-style','price']]
df_grouped=df_group.groupby(['drive-wheels','body-style'],as_index=False).mean()

df_pivot=df_grouped.pivot(index='drive-wheels',columns='body-style',values='price')

fig,ax=plt.subplots()
im=ax.pcolor(df_pivot,cmap='RdBu')

x_labels=df_pivot.columns
y_labels=df_pivot.index

ax.set_xticks(np.arange(df_pivot.shape[1])+0.5,minor=False)
ax.set_yticks(np.arange(df_pivot.shape[0])+0.5,minor=False)

ax.set_xticklabels(x_labels,minor=False)
ax.set_yticklabels(y_labels,minor=False)

plt.xticks(rotation=90)
fig.colorbar(im)
plt.show()

**Observation:** rwd convertibles and hardtops have the highest average price, while fwd and 4wd are much lower across most body styles, and the size of the gap varies by `body-style` . This shows that both variables work together, where the effect of `drive-wheels` depends on `body-style`.

## Pearson correlation for the numeric columns  
We test the numeric variables, and if they have a strong correlation with price and how certain we can be of this correlation.

In [ ]:
from scipy import stats
numeric_cols = [
    'symboling', 'normalized-losses', 'wheel-base', 'length', 'width',
    'height', 'curb-weight', 'engine-size', 'bore', 'stroke',
    'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg',
    'highway-mpg','num-of-doors', 'num-of-cylinders'
]
for i in numeric_cols:
 pearson_coef,p_value=stats.pearsonr(df[i],df['price'])
 print(f"{i}: coef={pearson_coef:.5f} ,P={p_value:.5f}")


**Observation:**  
`wheel-base`, `length`, `width`, `curb-weight`, `engine-size`, `bore` , `horsepower` , `city-mpg`, `highway-mpg`, and `num-of-cylinders` have a strong correlation and are statistically significant. So they will be added to the predictive model.

`symboling`, `stroke`, `compression-ratio`, `peak-rpm`, `num-of-doors`, `normalized-losses`, and `height`  are not statistically significant so they will be excluded from the predictive model.


## Developing a ridge regression model
we will be using the feature variables that we found to be good predictors in the model. the ridge regression helps us in fitting the model and controlling overfitting.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict

num_cols = ['wheel-base', 'length', 'width', 'curb-weight', 'engine-size', 'bore', 'horsepower', 'city-mpg', 'highway-mpg', 'num-of-cylinders']
scalar=StandardScaler()
df_num_scaled=pd.DataFrame(scalar.fit_transform(df[num_cols]),columns=num_cols,index=df.index)

x=pd.concat([df_num_scaled,df[new_col]],axis=1)
y=df['price']

pipe = Pipeline([('poly', PolynomialFeatures()),('model', Ridge())])

parameter_grid={'poly__degree':[1,2,3,4,5],'model__alpha':[0.001, 0.01, 0.1, 1, 10, 100, 1000]}

grid=GridSearchCV(estimator=pipe, param_grid=parameter_grid,cv=5,scoring='r2')

grid.fit(x,y)

print('Best parameters:', grid.best_params_)
print('Best cross validation r-squared:' ,grid.best_score_)
best_model=grid.best_estimator_

## Hyperparameter tuning: Polynomial Degree and Ridge alpha via GridSearchCV  
We use `GridSearchCV` to select the best combination that works under cross validation.  

Our features variable combines the continous variables that we performed pearson correlation on `wheel-base`, `length`, `width`, `curb-weight`, `engine-size`, `bore`, `horsepower`, `city-mpg`, `highway-mpg`, and `num-of-cylinders` with the one-hot encoded categorical variables inside of `new_col`.

**StandardScalar:** It was done separtely so the one hot encoded variables dont get standardized in the pipeline.

pipline contains two steps:   
**PolynomialFeatures:** helps us deal with the curved data set.  
**Ridge:** allows us to control how strongly large coefficiants are penalized.

## Model Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error,r2_score
yhat_cv=cross_val_predict(best_model,x,y,cv=5)
r2=r2_score(y,yhat_cv)
mse=mean_squared_error(y,yhat_cv)
rmse=np.sqrt(mse)

print('------Model Performance------')
print(f'r-squared:{r2:.3f}')
print(f'MSE:{mse:.3f}')
print(f'RMSE:{rmse:.3f}')

In [ ]:
import seaborn as sns
sns.kdeplot(y,color='r',label='Actual values')
sns.kdeplot(yhat_cv,color='b',label='Fitted values')
plt.title('Actual vs Predicted Price Distribution')
plt.xlabel('Price')
plt.legend()
plt.show


## Conclusion  
-We have managed to find the statistically significant categorical variables through anova, as well as the statistically significant numeric variables through pearson corrleation.  

-We found out that `drive-wheels` and `body-style` can both have an effect on price that differs from the effect of each of them individually through a heat map.  

-Ridge regression (α = 1000) gave the best validation performance, reducing overfitting.  

-The best polnomial degree for fitting the data (2) which helped improve the model.

-We developed a Ridge regression model to predict car prices using the UCI Automobile dataset. We built complex model that included both strong numeric predictors and one-hot encoded categorical variables. 

